# **Project Name**    - Brain Tumor MRI Image Classification (Custom CNN vs. Transfer Learning)

# **How to run this notebook**

1. **Colab:** *Runtime -> Change runtime type -> GPU (T4)*. Upload `drive-download-20261009T175229Z-1-001.zip` (the dataset) to the Colab file panel, or mount Google Drive and put it in `MyDrive`. **Local:** put the unzipped dataset in `./data` (folders `train/ valid/ test/`).
2. Set `FAST_MODE = True` in the configuration cell for a ~5-minute smoke test of the whole notebook (1-2 epochs per model; results will be poor). Leave it `False` for the real run (about 30-45 min on a T4 GPU).
3. *Runtime -> Run all.* The notebook is written to execute top to bottom without manual steps. Sections 1-6 and the classical baseline need no TensorFlow/GPU; Section 7 trains the deep models.
4. Outputs: trained models in `models/*.h5`, charts in `reports/eda/`, comparison table in `reports/model_comparison.csv`.

# **Project Summary -**

Brain tumours are diagnosed from MRI scans, and a radiologist has to decide not only *whether* a tumour is present but *which type* it is - glioma, meningioma or pituitary tumour - because treatment differs sharply between them. This project builds and compares deep-learning classifiers that assign an MRI slice to one of four classes (glioma, meningioma, pituitary tumour, no tumour), and wraps the best one in a Streamlit web application for real-time predictions.

**Data.** The dataset contains 2,443 labelled 640x640 RGB MRI slices (805 glioma, 610 pituitary, 545 meningioma, 483 no tumour), pre-split into train/valid/test folders. I audited it before modelling. Labels in the CSV files match the folder structure, there are no unreadable or byte-identical files, and class proportions are consistent across splits. Two problems turned up, though. **(1) Split leakage:** the Roboflow export created flipped or shifted variants of the same original scan and scattered them across splits. 97 of 502 validation images and 67 of 246 test images share their original scan with another split - and every one of them is a glioma, so 84% of the official test gliomas are effectively training images. **(2) Acquisition shortcuts:** the classes differ systematically in orientation, MRI sequence and framing; a random-forest that sees only seven global brightness/texture numbers (no spatial information at all) reaches about 87% accuracy on the official test set, against 33% for always guessing the biggest class. A CNN can exploit the same shortcuts, so its accuracy must be read with care.

**Method.** To obtain trustworthy numbers I re-split all images 70/15/15, stratified by class and *grouped by original scan*, so no scan appears in two splits (the official split remains available as an option, evaluated both in full and on its leak-free subset). Images are resized to 224x224, scaled to [0, 1] and augmented (rotation, flips, zoom, shifts, brightness and contrast jitter; black fill so no unrealistic mirrored background is created). Class imbalance is handled with class weights. I compare a classical HOG + SVM baseline, a custom CNN (four double-convolution blocks with batch normalisation and dropout, tuned with a small random search) and three ImageNet-pretrained backbones - MobileNetV2, ResNet50V2 and EfficientNetB0 - trained in two stages (new head first, then fine-tuning the top backbone layers at a low learning rate). Training uses EarlyStopping, ModelCheckpoint and ReduceLROnPlateau.

**Evaluation and deployment.** Models are compared on accuracy, macro precision/recall/F1, confusion matrices, a 95% confidence interval for accuracy, size and latency; the deployment model is chosen by a stated rule (best macro-F1, preferring the smaller model when scores are within noise). Occlusion-sensitivity maps show which regions drive a prediction, and an error analysis lists the mistakes. The best model is saved as `.h5`, reloaded as a sanity check, and served through the Streamlit app, which shows the predicted tumour type, per-class confidence, an optional explanation map and clear warnings about low confidence. The final metrics are generated by the notebook itself (Section 7.6 and the Conclusion), so they always correspond to the run that produced the saved models.

# **GitHub Link -**

Provide your GitHub Link here. *(push the repository folder `brain-tumor-mri-classification/`, then paste the URL)*

# **Problem Statement**

**Build a deep-learning system that classifies brain MRI slices into glioma, meningioma, pituitary tumour or no tumour, compare a custom CNN with transfer-learning models, choose the most accurate, efficient and reliable one, and deploy it in a Streamlit app that returns the predicted tumour type with confidence scores - while making sure the reported performance is not inflated by data leakage or dataset shortcuts.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries (TensorFlow is imported later, in Section 7, so Sections 1-6 run without it)
import os, re, json, time, random, shutil, zipfile, warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy import stats

from skimage.filters import sobel
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

### Configuration
All constants are in one place. `SPLIT_MODE` chooses the evaluation protocol (`"grouped"` = leak-free re-split, recommended; `"official"` = the dataset's own split). `FAST_MODE` shrinks every training budget for a quick end-to-end check.

In [ ]:
# ---------------------------------------------------------------- configuration
SEED = 42
IMG_SIZE = (224, 224)            # (height, width) given to every model
INPUT_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]   # index == integer label
DISPLAY_NAMES = {"glioma": "Glioma", "meningioma": "Meningioma",
                 "no_tumor": "No tumor", "pituitary": "Pituitary tumor"}
MODEL_NAMES = ["custom_cnn", "mobilenetv2", "resnet50v2", "efficientnetb0"]

SPLIT_MODE = "grouped"           # "grouped" (leak-free, recommended) or "official"
FAST_MODE = False                # True = tiny training budget, only to test the pipeline

# Paths
ZIP_NAME = "drive-download-20261009T175229Z-1-001.zip"
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
REPORTS_DIR = Path("reports")
for folder in (MODELS_DIR, REPORTS_DIR / "eda"):
    folder.mkdir(parents=True, exist_ok=True)

# Used by the helper functions below
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
_SOURCE_RE = re.compile(r"^Tr-([a-z]+)_(\d+)_", re.IGNORECASE)
FEATURE_COLUMNS = ["brightness", "contrast", "skewness", "entropy",
                   "foreground_frac", "edge_density", "lr_symmetry", "file_kb"]
TRANSFER_BACKBONES = {   # name -> (keras.applications class, how [0,1] input is rescaled)
    "mobilenetv2": ("MobileNetV2", "minus1_1"),
    "resnet50v2": ("ResNet50V2", "minus1_1"),
    "efficientnetb0": ("EfficientNetB0", "raw255"),
}
PAL = dict(zip(CLASS_NAMES, sns.color_palette("colorblind", 4)))

random.seed(SEED); np.random.seed(SEED)

### Helper functions
These are copied verbatim from the repository's `src/` package (`data.py`, `eda.py`, `evaluate.py`, `explain.py`, `inference.py`), so the notebook and the deployed code cannot drift apart. Each function has a docstring explaining its logic.

In [ ]:
# src/data.py

def source_id_from_filename(filename: str) -> str:
    """Return the id of the original scan, e.g. 'gl_0352'.

    Falls back to the full file name when the pattern does not match, which
    means "treat this image as its own scan" (never creates false leakage).
    """
    match = _SOURCE_RE.match(filename)
    if not match:
        return filename
    return f"{match.group(1).lower()}_{match.group(2)}"


def build_dataframe(data_dir: str | Path) -> pd.DataFrame:
    """Index every image under data_dir/{train,valid,test}/{class}/ into a DataFrame."""
    data_dir = Path(data_dir)
    rows = []
    for split in ("train", "valid", "test"):
        for label, class_name in enumerate(CLASS_NAMES):
            folder = data_dir / split / class_name
            if not folder.is_dir():
                continue
            for path in sorted(folder.iterdir()):
                if path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append(
                        {
                            "split": split,
                            "class_name": class_name,
                            "label": label,
                            "filename": path.name,
                            "path": str(path),
                            "source_id": source_id_from_filename(path.name),
                        }
                    )
    if not rows:
        raise FileNotFoundError(
            f"No images found under {data_dir}. Expected {data_dir}/train/<class>/*.jpg"
        )
    return pd.DataFrame(rows)


def flag_leakage(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean column `leak_free`.

    * train rows are always leak-free (they define what the model has seen);
    * a valid row is leak-free if its original scan is NOT in train;
    * a test row is leak-free if its original scan is in NEITHER train nor valid.
    """
    df = df.copy()
    train_ids = set(df.loc[df["split"] == "train", "source_id"])
    valid_ids = set(df.loc[df["split"] == "valid", "source_id"])

    in_train = df["source_id"].isin(train_ids)
    in_valid = df["source_id"].isin(valid_ids)

    df["leak_free"] = True
    df.loc[df["split"] == "valid", "leak_free"] = ~in_train[df["split"] == "valid"]
    df.loc[df["split"] == "test", "leak_free"] = ~(in_train | in_valid)[df["split"] == "test"]
    return df


def make_grouped_split(df: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    """Re-split ALL images ~70/15/15 so no original scan is shared between splits.

    Stratified by class and grouped by `source_id` (StratifiedGroupKFold), which keeps
    every augmented variant of a scan together. The dataset's own split is kept in
    column `official_split`; `split` holds the new assignment and `leak_free` is True
    everywhere. Note: this removes duplicate-scan leakage, but the dataset has no patient
    id, so different slices of one patient can still land in different splits.
    """
    from sklearn.model_selection import StratifiedGroupKFold

    df = df.copy().reset_index(drop=True)
    df["official_split"] = df["split"]

    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)   # 1/7 -> test
    rest_idx, test_idx = next(outer.split(df, df["label"], df["source_id"]))
    rest = df.iloc[rest_idx]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)   # 1/6 of rest -> valid
    _, val_pos = next(inner.split(rest, rest["label"], rest["source_id"]))

    df["split"] = "train"
    df.loc[df.index[test_idx], "split"] = "test"
    df.loc[rest.index[val_pos], "split"] = "valid"
    df["leak_free"] = True
    return df


def load_dataframe(data_dir, split_mode: str = "grouped") -> pd.DataFrame:
    """Index the dataset and apply the chosen evaluation protocol.

    split_mode = "grouped"  : leak-free re-split by original scan (default, recommended)
    split_mode = "official" : the dataset's own train/valid/test, with a `leak_free` flag
                              so results can also be reported on the clean subset
    """
    df = flag_leakage(build_dataframe(data_dir))
    if split_mode == "grouped":
        return make_grouped_split(df)
    if split_mode == "official":
        return df
    raise ValueError("split_mode must be 'grouped' or 'official'")


def get_class_weights(labels: np.ndarray) -> dict[int, float]:
    """'Balanced' class weights: n_samples / (n_classes * count_of_class)."""
    labels = np.asarray(labels)
    classes, counts = np.unique(labels, return_counts=True)
    weights = len(labels) / (len(classes) * counts)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def make_dataset(paths, labels, training: bool, batch_size: int = BATCH_SIZE,
                 img_size=IMG_SIZE, augmentation=None):
    """Build a batched tf.data pipeline: decode -> resize -> [0,1] -> (augment).

    * `antialias=True` makes tf.image.resize behave like PIL's BILINEAR resize,
      so images preprocessed in the Streamlit app match training preprocessing.
    * Decoded images are cached in RAM (about 150 KB each at 224x224x3).
    * `augmentation` (a Keras Sequential) is applied after batching and only
      when training=True, so every image in a batch gets its own random transform.
    """
    import tensorflow as tf  # imported lazily so this module is usable without TF

    paths = np.asarray(paths, dtype=str)
    labels = np.asarray(labels, dtype=np.int32)

    def load(path, label):
        raw = tf.io.read_file(path)
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size, antialias=True)
        img = tf.cast(img, tf.float32) / 255.0
        img = tf.clip_by_value(img, 0.0, 1.0)  # resize ringing can overshoot slightly
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    if training and augmentation is not None:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# src/eda.py

def load_gray(path: str, size: int = 128) -> np.ndarray:
    """Load an image as float32 grayscale in [0, 1], resized to size x size."""
    img = Image.open(path).convert("L").resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def image_features(gray: np.ndarray, fg_threshold: float = 0.08) -> dict:
    """Summarise one grayscale image (values in [0, 1]).

    brightness       mean pixel intensity
    contrast         standard deviation of intensities
    skewness         asymmetry of the intensity histogram
    entropy          Shannon entropy of the 64-bin histogram (texture richness)
    foreground_frac  share of pixels brighter than fg_threshold (head/brain area)
    edge_density     mean Sobel gradient magnitude (structure / detail)
    lr_symmetry      correlation between the image and its left-right mirror;
                     healthy axial slices are close to symmetric, large masses
                     reduce it
    """
    hist, _ = np.histogram(gray, bins=64, range=(0, 1))
    p = hist / hist.sum()
    p = p[p > 0]
    flat = gray.ravel()
    mirrored = gray[:, ::-1].ravel()
    sym = np.corrcoef(flat, mirrored)[0, 1] if flat.std() > 0 else 0.0
    return {
        "brightness": float(gray.mean()),
        "contrast": float(gray.std()),
        "skewness": float(stats.skew(flat)),
        "entropy": float(-(p * np.log2(p)).sum()),
        "foreground_frac": float((gray > fg_threshold).mean()),
        "edge_density": float(sobel(gray).mean()),
        "lr_symmetry": float(sym),
    }


def compute_image_features(df: pd.DataFrame, size: int = 128) -> pd.DataFrame:
    """Return df plus one column per feature in `image_features` and file_kb."""
    feats = []
    for path in df["path"]:
        feats.append(image_features(load_gray(path, size)))
    out = pd.concat([df.reset_index(drop=True), pd.DataFrame(feats)], axis=1)
    out["file_kb"] = [round(os.path.getsize(p) / 1024, 1) for p in out["path"]]
    return out

In [ ]:
# src/evaluate.py

def wilson_ci(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """95% Wilson score interval for a proportion (better than +-1.96*se for small n)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (float(centre - half), float(centre + half))


def evaluate_predictions(y_true, y_prob, class_names) -> dict:
    """Full metric set from true integer labels and predicted class probabilities."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_prob).argmax(axis=1)
    labels = list(range(len(class_names)))

    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    _, _, w_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    n_correct = int((y_true == y_pred).sum())
    lo, hi = wilson_ci(n_correct, len(y_true))

    return {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "accuracy_ci": [lo, hi],
        "macro_precision": float(prec.mean()),
        "macro_recall": float(rec.mean()),
        "macro_f1": float(f1.mean()),
        "weighted_f1": float(w_f1),
        "per_class": {
            name: {"precision": float(prec[i]), "recall": float(rec[i]),
                   "f1": float(f1[i]), "support": int(support[i])}
            for i, name in enumerate(class_names)
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def plot_confusion_matrix(cm, class_names, title="Confusion matrix", ax=None):
    """Heat-map showing counts, with row-normalised percentages (= per-class recall)."""
    cm = np.asarray(cm)
    row_pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.imshow(row_pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            colour = "white" if row_pct[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm[i, j]}\n{row_pct[i, j]:.0%}", ha="center",
                    va="center", color=colour, fontsize=9)
    ax.set_xticks(range(len(class_names)), class_names, rotation=30, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    return ax


def plot_history(history: dict, title: str = "", finetune_start: int | None = None):
    """Training vs validation accuracy and loss. Dashed line marks where fine-tuning began."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, key, label in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        ax.plot(history[key], label="train")
        ax.plot(history[f"val_{key}"], label="validation")
        if finetune_start:
            ax.axvline(finetune_start - 0.5, ls="--", c="grey", lw=1, label="fine-tuning starts")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(f"{title} {label}".strip())
        ax.legend()
        ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def predict_probs(model, ds):
    """Run a model over an (unshuffled) tf.data pipeline -> (y_true, y_prob)."""
    y_true = np.concatenate([y.numpy() for _, y in ds])
    y_prob = model.predict(ds, verbose=0)
    return y_true, y_prob


def measure_latency_ms(model, input_shape=(224, 224, 3), n_runs: int = 30) -> float:
    """Median single-image inference time in milliseconds (after one warm-up call)."""
    x = np.random.default_rng(0).random((1, *input_shape), dtype=np.float32)
    model.predict(x, verbose=0)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(x, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def compare_models(results: dict[str, dict]) -> pd.DataFrame:
    """One row per model. `results[name]` must hold keys:
    'all' and 'leak_free' (outputs of evaluate_predictions on the whole test set and on
    its leak-free subset) plus 'params', 'size_mb', 'latency_ms'.

    Headline columns come from the leak-free evaluation. Columns for the unfiltered test
    set are only added when the filter actually removed images (official split mode);
    with the grouped split both evaluations are identical and the extra columns are omitted.
    """
    show_all = any(r["all"]["n"] != r["leak_free"]["n"] for r in results.values())
    rows = []
    for name, r in results.items():
        row = {"model": name,
               "params_M": round(r["params"] / 1e6, 2),
               "size_MB": round(r["size_mb"], 1),
               "latency_ms": round(r["latency_ms"], 1)}
        if show_all:
            row["acc_all_test"] = round(r["all"]["accuracy"], 4)
            row["f1_all_test"] = round(r["all"]["macro_f1"], 4)
        row.update({
            "accuracy": round(r["leak_free"]["accuracy"], 4),
            "acc_ci_low": round(r["leak_free"]["accuracy_ci"][0], 4),
            "acc_ci_high": round(r["leak_free"]["accuracy_ci"][1], 4),
            "precision": round(r["leak_free"]["macro_precision"], 4),
            "recall": round(r["leak_free"]["macro_recall"], 4),
            "f1": round(r["leak_free"]["macro_f1"], 4),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)


def select_best_model(table: pd.DataFrame, tolerance: float = 0.005) -> str:
    """Pick the deployment model.

    Rule: highest macro-F1 on the clean (leak-free) test set; among models within
    `tolerance` of that score, prefer the one with the fewest parameters
    (cheaper to serve, and the difference is inside the noise anyway).
    """
    top = table["f1"].max()
    contenders = table[table["f1"] >= top - tolerance]
    return str(contenders.sort_values("params_M").iloc[0]["model"])

In [ ]:
# src/explain.py

def occlusion_sensitivity(predict_fn: Callable[[np.ndarray], np.ndarray],
                          image: np.ndarray, target_class: int,
                          patch: int = 32, stride: int = 16,
                          fill_value: float = 0.0) -> np.ndarray:
    """Return an (H, W) heat-map in [0, 1]; higher = occluding here hurts more.

    predict_fn : maps a batch (N, H, W, 3) in [0, 1] to class probabilities (N, C)
    image      : (H, W, 3) float array in [0, 1]
    """
    h, w, _ = image.shape
    base = float(predict_fn(image[None])[0, target_class])

    ys = list(range(0, max(h - patch, 0) + 1, stride))
    xs = list(range(0, max(w - patch, 0) + 1, stride))
    if ys[-1] + patch < h:
        ys.append(h - patch)          # make sure the bottom/right edges get covered
    if xs[-1] + patch < w:
        xs.append(w - patch)

    batch, coords = [], []
    for y in ys:
        for x in xs:
            occluded = image.copy()
            occluded[y:y + patch, x:x + patch, :] = fill_value
            batch.append(occluded)
            coords.append((y, x))
    probs = predict_fn(np.stack(batch))[:, target_class]

    heat = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)
    for (y, x), p in zip(coords, probs):
        heat[y:y + patch, x:x + patch] += base - p
        count[y:y + patch, x:x + patch] += 1
    heat = heat / np.maximum(count, 1)
    heat = np.clip(heat, 0, None)      # only keep evidence FOR the class
    return heat / heat.max() if heat.max() > 0 else heat


def overlay_heatmap(image: np.ndarray, heat: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Blend a 'jet' colour-mapped heat-map over the image; returns uint8 RGB."""
    from matplotlib import colormaps
    colour = colormaps["jet"](heat)[..., :3]
    blended = (1 - alpha) * image + alpha * colour
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)

In [ ]:
# src/inference.py

def preprocess_pil(img: Image.Image, img_size=IMG_SIZE) -> np.ndarray:
    """PIL image (any mode) -> float32 array (H, W, 3) in [0, 1]."""
    img = img.convert("RGB")                       # handles grayscale / RGBA / palette
    img = img.resize((img_size[1], img_size[0]), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def describe_prediction(probs: np.ndarray) -> dict:
    """Turn a probability vector into something display-ready."""
    order = np.argsort(probs)[::-1]
    top = int(order[0])
    return {
        "label": CLASS_NAMES[top],
        "display": DISPLAY_NAMES[CLASS_NAMES[top]],
        "confidence": float(probs[top]),
        "ranking": [(DISPLAY_NAMES[CLASS_NAMES[i]], float(probs[i])) for i in order],
    }

### Dataset Loading

In [ ]:
# Locate the dataset: use ./data if present, otherwise unzip the uploaded zip file.
if not (DATA_DIR / "train").exists():
    for candidate in (Path(ZIP_NAME), Path("/content") / ZIP_NAME,
                      Path("/content/drive/MyDrive") / ZIP_NAME):
        if candidate.exists():
            print("Extracting", candidate)
            with zipfile.ZipFile(candidate) as zf:
                zf.extractall(DATA_DIR)
            break
    else:
        raise FileNotFoundError(
            f"Dataset not found. Put {ZIP_NAME} next to this notebook (or in /content), "
            "or unzip it into ./data so that ./data/train exists.")

# Index every image. Here we always use the OFFICIAL split so the audit below is about the
# data exactly as delivered; the evaluation protocol (SPLIT_MODE) is applied in Section 6.
df_official = flag_leakage(build_dataframe(DATA_DIR))
print(f"Indexed {len(df_official):,} images from {DATA_DIR.resolve()}")

### Dataset First View

In [ ]:
# Dataset First Look
df_official.head()

In [ ]:
# A quick visual look: 5 random training images per class
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for r, cls in enumerate(CLASS_NAMES):
    sample = df_official[(df_official.split == "train") & (df_official.class_name == cls)].sample(5, random_state=1)
    for c, path in enumerate(sample.path):
        axes[r, c].imshow(Image.open(path).convert("RGB"))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(DISPLAY_NAMES[cls], loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Rows (images):", df_official.shape[0], "| Columns (metadata):", df_official.shape[1])
counts = pd.crosstab(df_official.split, df_official.class_name, margins=True)[CLASS_NAMES + ["All"]]
display(counts.loc[["train", "valid", "test", "All"]])
print("README says: train 1695, valid 502, test 246 ->",
      "matches" if (counts.loc[["train", "valid", "test"], "All"].tolist() == [1695, 502, 246]) else "DIFFERS")

### Dataset Information

In [ ]:
# Dataset Info
df_official.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
# (a) byte-identical files
import hashlib
df_official["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df_official.path]
print("Byte-identical duplicate files:", int(df_official.md5.duplicated().sum()))

# (b) different files that are variants (flip/shift) of the SAME original scan
per_scan = df_official.groupby("source_id").size()
print("Original scans:", len(per_scan), "| scans with more than one image:", int((per_scan > 1).sum()))
split_count = df_official.groupby("source_id").split.nunique()
print("Scans that appear in more than one split:", int((split_count > 1).sum()))

# (c) how much of valid/test is a variant of something in another split?
leak = (df_official[df_official.split != "train"]
        .groupby("split").leak_free.agg(images="size", leaked=lambda s: int((~s).sum())))
leak["leaked_%"] = (100 * leak.leaked / leak.images).round(1)
display(leak)
print("Classes of the leaked images:")
display(df_official[~df_official.leak_free].groupby(["split", "class_name"]).size().rename("leaked images").to_frame())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count (metadata) + unreadable images + CSV-vs-folder label check
print("Null values in metadata:\n", df_official.isna().sum().to_string(), sep="")

unreadable = []
for p in df_official.path:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        unreadable.append(p)
n_unreadable = len(unreadable)
print("\nUnreadable / corrupt images:", n_unreadable)

csv_mismatch = 0
for split in ("train", "valid", "test"):
    csv = pd.read_csv(DATA_DIR / split / "_classes.csv", skipinitialspace=True)
    csv_label = csv.iloc[:, 1:].values.argmax(1)
    names = [c.strip().lower().replace(" ", "_") for c in csv.columns[1:]]
    from_csv = pd.Series([names[i] for i in csv_label], index=csv.filename.str.strip())
    from_folder = df_official[df_official.split == split].set_index("filename").class_name
    joined = pd.concat([from_csv.rename("csv"), from_folder.rename("folder")], axis=1)
    csv_mismatch += int((joined.csv != joined.folder).sum())
print("Images whose CSV label disagrees with their folder (or are missing from one):", csv_mismatch)

In [ ]:
# Visualizing the missing values / data-quality problems
quality = pd.Series({"null metadata cells": int(df_official.isna().sum().sum()),
                     "unreadable images": n_unreadable,
                     "CSV/folder label mismatches": csv_mismatch,
                     "byte-identical duplicates": int(df_official.md5.duplicated().sum()),
                     "images leaked across splits": int((~df_official.leak_free).sum())})
ax = quality.plot(kind="barh", color=["#4c9a2a"] * 4 + ["#c0392b"], figsize=(7, 3))
ax.set_xlabel("Count"); ax.set_title("Data-quality checks (green = clean, red = problem)")
for i, v in enumerate(quality.values):
    ax.text(v + 1, i, str(v), va="center")
plt.tight_layout(); plt.show()

### What did you know about your dataset?

* **Size & structure:** 2,443 RGB JPEG slices, all 640x640 pixels, in `train` (1,695), `valid` (502) and `test` (246), one folder per class; four classes: glioma (805), pituitary (610), meningioma (545), no tumour (483). Glioma is the largest class (33%), no tumour the smallest (20%) - a mild 1.7 : 1 imbalance.
* **Clean on the basics:** no null metadata, no unreadable files, no byte-identical duplicates, and the `_classes.csv` labels agree with the folders for every image.
* **Not clean on the splits:** many images are flipped/shifted variants of one original scan (file names share the `Tr-<class>_<id>` part). 164 validation/test images share a scan with another split - **97 of 502 validation (19%) and 67 of 246 test (27%)** - and **all of them are gliomas**: 60% of validation gliomas and 84% of test gliomas. Evaluating on the official test set therefore partly measures memorisation, and almost all of that inflation lands on one class.
* **Consequence:** the headline protocol in this project re-splits by original scan (Section 6.8), and the official split can still be evaluated with a leak-free subset.

## ***2. Understanding Your Variables***

In [ ]:
# Compute image-level variables from the pixels (about 15-30 s). They are used for EDA, the
# hypothesis tests and the baseline only - the deep models learn their own features.
t0 = time.time()
feat = compute_image_features(df_official)
print(f"Computed {len(FEATURE_COLUMNS)} variables for {len(feat):,} images in {time.time() - t0:.0f}s")

# Dataset Columns
print(list(feat.columns))

In [ ]:
# Dataset Describe
feat[FEATURE_COLUMNS].describe().T.round(3)

### Variables Description

| Variable | Type | Meaning |
|---|---|---|
| `split` | categorical | Official split the image was delivered in: train / valid / test |
| `class_name`, `label` | categorical / int | Target: glioma, meningioma, no_tumor, pituitary -> 0, 1, 2, 3 |
| `source_id` | categorical | Id of the *original scan* parsed from the file name (e.g. `gl_0352`); several images can share one |
| `leak_free` | bool | False if the image's original scan also occurs in train (or, for test, in valid) |
| `brightness` | float 0-1 | Mean pixel intensity |
| `contrast` | float | Standard deviation of intensities |
| `skewness` | float | Asymmetry of the intensity histogram (large = mostly dark with a few bright regions) |
| `entropy` | float (bits) | Shannon entropy of the intensity histogram - texture richness |
| `foreground_frac` | float 0-1 | Share of pixels brighter than 0.08 - how much of the frame the head fills |
| `edge_density` | float | Mean Sobel gradient magnitude - amount of structural detail |
| `lr_symmetry` | float -1..1 | Correlation of the image with its left-right mirror |
| `file_kb` | float | JPEG file size in KB - a proxy for image detail / compression |

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
pd.DataFrame({"unique values": feat.nunique(),
              "example": [feat[c].iloc[0] if c != "path" else "..." for c in feat.columns]})

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
feat["class_display"] = feat.class_name.map(DISPLAY_NAMES)
feat["split"] = pd.Categorical(feat.split, ["train", "valid", "test"], ordered=True)
feat["leak_status"] = np.where(feat.leak_free, "clean", "same scan as another split")

# Summary: per-class medians of every variable - the table the charts below explain
display(feat.groupby("class_name")[FEATURE_COLUMNS].median().round(3))

### What all manipulations have you done and insights you found?

* Parsed the **original-scan id** from each file name and flagged every validation/test image whose scan also occurs in another split (`leak_free`).
* Computed eight pixel-level variables per image (on a 128x128 grayscale copy for speed) and a readable class label.
* No rows were dropped or imputed: there is nothing missing or corrupt.
* **First insight from the medians:** the classes differ strongly in basic statistics - glioma images are darkest (median brightness 0.10 vs 0.22 for no tumour) and fill the least of the frame. Those differences come from *how the scans were acquired*, not (only) from tumour biology, which is why Section 4 spends several charts on them.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

Charts follow the **UBM** rule: Univariate (1-4, 8, 9, 13), Bivariate (2, 3, 5-7, 10-12), Multivariate (14-17). Every chart is also saved to `reports/eda/`.

In [ ]:
def savefig(name):
    """Save the current figure to reports/eda/<name>.png and show it."""
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "eda" / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

LABELS = [DISPLAY_NAMES[c] for c in CLASS_NAMES]

#### Chart - 1 - Class distribution

In [ ]:
# Chart - 1 visualization code
counts = feat.class_name.value_counts().reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(LABELS, counts.values, color=[PAL[c] for c in CLASS_NAMES])
for i, v in enumerate(counts.values):
    ax.text(i, v + 8, f"{v}\n({v / len(feat):.1%})", ha="center")
ax.set_ylabel("Number of images"); ax.set_ylim(0, counts.max() * 1.18)
ax.set_title("Class distribution - all 2,443 images")
savefig("01_class_distribution")

##### 1. Why did you pick the specific chart?

A bar chart is the simplest honest way to compare category sizes, and printing counts and percentages on the bars lets the reader see the imbalance exactly.

##### 2. What is/are the insight(s) found from the chart?

Glioma is the largest class (805 images, 33%), followed by pituitary (610, 25%) and meningioma (545, 22%); no tumour is the smallest (483, 20%). The largest class is only about 1.7x the smallest, so the imbalance is mild, not severe.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. A mild imbalance means a model can be trained without resampling, but plain accuracy would slightly favour glioma, so I use class weights during training and report macro-averaged precision/recall/F1. The risk to avoid is the opposite one: ignoring per-class recall would hide weak performance on the smaller 'no tumour' and meningioma classes, which in practice means missed or mislabelled patients.

#### Chart - 2 - Class mix per split

In [ ]:
# Chart - 2 visualization code
mix = (pd.crosstab(feat.split, feat.class_name, normalize="index")[CLASS_NAMES] * 100)
ax = mix.plot(kind="bar", figsize=(7.5, 4), color=[PAL[c] for c in CLASS_NAMES], width=0.8)
ax.set_ylabel("% of the split's images"); ax.set_xlabel("")
ax.set_xticklabels(["train (1,695)", "valid (502)", "test (246)"], rotation=0)
ax.legend(LABELS, title="Class", ncol=2); ax.set_title("Class proportions inside each official split")
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.0f", fontsize=8)
savefig("02_class_mix_per_split")

##### 1. Why did you pick the specific chart?

A grouped bar chart of *within-split percentages* answers 'is each split a fair sample of the whole?' directly, which counts alone cannot show because the splits have different sizes.

##### 2. What is/are the insight(s) found from the chart?

The class mix is almost identical in train, valid and test (glioma 32-33%, meningioma 21-26%, no tumour about 20%, pituitary 22-26%). The official split was therefore stratified, and a chi-square test (Section 5, Hypothesis 2) confirms the differences are within chance.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: validation and test scores are comparable in class composition to training, so a drop on the test set would not be an artefact of a different class mix. No negative signal here - the problem with the official split is not *which classes* are in each part but *which scans* (next chart).

#### Chart - 3 - Leakage across splits

In [ ]:
# Chart - 3 visualization code
held_out = feat[feat.split != "train"].copy()
held_out["status"] = np.where(held_out.leak_free, "clean", "variant of a scan in another split")
by_split = pd.crosstab(held_out.split, held_out.status).loc[["valid", "test"]]
by_class = held_out.groupby(["split", "class_name"]).leak_free.apply(lambda s: (~s).mean() * 100).unstack()[CLASS_NAMES].loc[["valid", "test"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
by_split.plot(kind="bar", stacked=True, ax=axes[0], color=["#4c9a2a", "#c0392b"], rot=0)
axes[0].set_title("Images in valid / test that duplicate a scan from another split")
axes[0].set_xlabel(""); axes[0].set_ylabel("Images"); axes[0].legend(title="")
for cont in axes[0].containers:
    axes[0].bar_label(cont, label_type="center", color="white", fontweight="bold")
by_class.plot(kind="bar", ax=axes[1], color=[PAL[c] for c in CLASS_NAMES], rot=0, width=0.8)
axes[1].set_title("...as % of each class (valid / test)"); axes[1].set_xlabel(""); axes[1].set_ylabel("% of class images that leak")
axes[1].legend(LABELS, title="Class")
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt="%.0f%%", fontsize=8)
savefig("03_leakage_across_splits")

##### 1. Why did you pick the specific chart?

Two panels answer two different questions: how big is the leakage per split (stacked bars) and *who* is affected (percentage per class). A single chart would hide that the problem is concentrated in one class.

##### 2. What is/are the insight(s) found from the chart?

97 of 502 validation images (19%) and 67 of 246 test images (27%) are flipped or shifted copies of a scan that is also in another split. **100% of them are gliomas**: 60% of validation gliomas and 84% of test gliomas. Meningioma, pituitary and no-tumour images do not leak at all. Only 13 of the 80 official test gliomas are genuinely unseen.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This is the most important business insight of the project, and it is a *warning*: a model scored on the official test set would look better than it is, especially on glioma, and a hospital relying on that number would over-trust the model. Fixing the protocol (grouped re-split, Section 6.8) costs nothing and makes every later metric honest - the cost of not fixing it would be a deployed model whose real-world glioma performance is unknown.

#### Chart - 4 - Image gallery

In [ ]:
# Chart - 4 visualization code
fig, axes = plt.subplots(4, 6, figsize=(14, 9.5))
for r, cls in enumerate(CLASS_NAMES):
    sample = feat[(feat.split == "train") & (feat.class_name == cls)].sample(6, random_state=7)
    for c, path in enumerate(sample.path):
        axes[r, c].imshow(Image.open(path).convert("RGB")); axes[r, c].axis("off")
    axes[r, 0].set_title(DISPLAY_NAMES[cls], loc="left", fontsize=13, fontweight="bold")
savefig("04_gallery")

##### 1. Why did you pick the specific chart?

For image data nothing replaces looking at the images: a gallery reveals orientation, contrast, framing and artefacts that no summary statistic captures.

##### 2. What is/are the insight(s) found from the chart?

The classes are not photographed alike. Glioma slices are mostly axial with a smaller head in the frame; pituitary slices are largely sagittal and coronal; meningioma slices are mostly coronal/axial with large bright masses; the no-tumour slices include T2/FLAIR-style contrast that looks different from the T1-style scans of the tumour classes. Tumours themselves appear as bright (contrast-enhanced) or dark masses of very different size and position.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Helpful: it shows the augmentation I should use (flips and small rotations are plausible; extreme zoom is not) and that the model must cope with several viewing planes. Risky: because orientation and contrast correlate with class, a network could learn 'which scanner/plane' instead of 'which tumour' - the next charts quantify this.

#### Chart - 5 - Brightness by class

In [ ]:
# Chart - 5 visualization code
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(data=feat, x="class_name", y="brightness", order=CLASS_NAMES, hue="class_name",
               palette=PAL, inner="quartile", legend=False, ax=ax, cut=0)
ax.set_xticklabels(LABELS); ax.set_xlabel(""); ax.set_ylabel("Mean pixel intensity (0-1)")
ax.set_title("Image brightness by class")
savefig("05_brightness_by_class")

##### 1. Why did you pick the specific chart?

A violin plot shows the whole distribution (shape, spread, quartiles) per class, so it exposes overlap between classes that a bar of averages would hide.

##### 2. What is/are the insight(s) found from the chart?

Median brightness rises from glioma (0.10) to meningioma (0.16), pituitary (0.20) and no tumour (0.22). Glioma is not only darker, its distribution is very narrow, so a brightness value alone says a lot about the class. A Kruskal-Wallis test (Section 5) gives p far below 0.001 with a large effect size.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Mixed. Positive: classes are separable, so learning should be easy. Negative: a difference this large and this tight is a signature of *acquisition* (scanner, sequence, windowing) rather than disease, so a model could reach high accuracy without learning tumour morphology and then fail on scans from another hospital. The project therefore reports a shortcut check (Chart 17) and recommends external validation before any clinical use.

#### Chart - 6 - Contrast by class

In [ ]:
# Chart - 6 visualization code
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=feat, x="class_name", y="contrast", order=CLASS_NAMES, hue="class_name",
            palette=PAL, legend=False, ax=ax, fliersize=2)
ax.set_xticklabels(LABELS); ax.set_xlabel(""); ax.set_ylabel("Standard deviation of intensities")
ax.set_title("Image contrast by class")
savefig("06_contrast_by_class")

##### 1. Why did you pick the specific chart?

Box plots give a compact five-number comparison plus outliers and are the standard way to compare a numeric variable across a few groups.

##### 2. What is/are the insight(s) found from the chart?

No-tumour images have the highest median contrast (0.21) and by far the widest spread (a mix of T1, T2 and FLAIR-style scans), followed by meningioma (0.18, with many high outliers from very bright contrast-enhanced masses), pituitary (0.15) and glioma (0.14, the narrowest).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Contrast is useful input but again partly an acquisition effect. For the business case the practical lesson is to keep *per-image* intensity normalisation out of the pipeline: rescaling every image to the same contrast would erase real signal (bright enhancing masses), while the brightness/contrast augmentation I use is mild enough to improve robustness to scanner differences without destroying it.

#### Chart - 7 - Head area in frame

In [ ]:
# Chart - 7 visualization code
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(data=feat, x="class_name", y="foreground_frac", order=CLASS_NAMES, hue="class_name",
               palette=PAL, inner="quartile", legend=False, ax=ax, cut=0)
ax.set_xticklabels(LABELS); ax.set_xlabel(""); ax.set_ylabel("Share of non-black pixels")
ax.set_title("How much of the frame the head fills")
savefig("07_foreground_by_class")

##### 1. Why did you pick the specific chart?

The share of non-black pixels is a direct, interpretable measure of framing/zoom, and a violin plot shows whether classes occupy distinct ranges.

##### 2. What is/are the insight(s) found from the chart?

Pituitary images fill the most frame (median 0.67), then no tumour (0.58), meningioma (0.49) and glioma (0.40). Different classes were therefore cropped or zoomed differently - consistent with the gallery, where glioma heads look smaller.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This justifies resizing every image to a fixed 224x224 *without* cropping (cropping would cut tumours near the edge) and applying random zoom and translation, so the network does not rely on head size. It is also another sign the dataset was assembled from heterogeneous sources, which limits how far results generalise.

#### Chart - 8 - Intensity histograms

In [ ]:
# Chart - 8 visualization code
rng = np.random.default_rng(0)
bins = np.linspace(0, 1, 33)
fig, ax = plt.subplots(figsize=(8, 4.5))
for cls in CLASS_NAMES:
    paths = feat[feat.class_name == cls].path.sample(80, random_state=0)
    hist = np.mean([np.histogram(load_gray(p, 96), bins=bins)[0] / 96**2 for p in paths], axis=0)
    ax.plot((bins[:-1] + bins[1:]) / 2, hist, label=DISPLAY_NAMES[cls], color=PAL[cls], lw=2)
ax.set_yscale("log"); ax.set_xlabel("Pixel intensity"); ax.set_ylabel("Share of pixels (log scale)")
ax.set_title("Average intensity histogram per class (80 images each)"); ax.legend()
savefig("08_intensity_histograms")

##### 1. Why did you pick the specific chart?

Overlaid average histograms show, in one picture, where each class puts its pixel mass; a log axis keeps the tiny share of bright pixels visible next to the huge black background.

##### 2. What is/are the insight(s) found from the chart?

In every class most pixels are near black (the background), but the curves separate in the mid and high intensities: no-tumour and meningioma images keep more pixels in the mid/high range, glioma images drop off fastest. The histograms confirm numerically what the brightness chart showed.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This supports using simple [0, 1] scaling (all classes share the same range, nothing saturates) and tells us not to rely on thresholds or hand-made intensity rules: the distributions overlap too much in the mid range for a rule to be reliable, so a learned model is justified.

#### Chart - 9 - Average image per class

In [ ]:
# Chart - 9 visualization code
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, cls in zip(axes, CLASS_NAMES):
    paths = feat[feat.class_name == cls].path.sample(150, random_state=0)
    ax.imshow(np.mean([load_gray(p, 128) for p in paths], axis=0), cmap="gray", vmin=0, vmax=0.6)
    ax.set_title(DISPLAY_NAMES[cls]); ax.axis("off")
savefig("09_mean_image_per_class")

##### 1. Why did you pick the specific chart?

A pixel-wise mean image summarises the 'typical' framing and anatomy of a class and makes systematic differences in position and scale visible at a glance.

##### 2. What is/are the insight(s) found from the chart?

The average images are blurry heads, as expected when scans differ in position and plane, but their outlines differ: the pituitary and no-tumour averages cover a larger area of the frame than the glioma average. No class average shows a sharp tumour - tumours vary too much in location to survive averaging.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: tumours are not at a fixed location, so a fully convolutional network with global pooling (translation-tolerant) is the right design, and position-based augmentation is safe. The differences in head outline are a reminder of the framing bias discussed in Chart 7.

#### Chart - 10 - Brightness vs edge density

In [ ]:
# Chart - 10 visualization code
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.scatterplot(data=feat, x="brightness", y="edge_density", hue="class_name", hue_order=CLASS_NAMES,
                palette=PAL, s=14, alpha=0.55, ax=ax)
ax.legend(title="Class", labels=LABELS); ax.set_xlabel("Mean brightness"); ax.set_ylabel("Edge density (mean Sobel magnitude)")
ax.set_title(f"Brightness vs. structural detail (r = {feat.brightness.corr(feat.edge_density):.2f})")
savefig("10_brightness_vs_edges")

##### 1. Why did you pick the specific chart?

A scatter plot is the right tool for two numeric variables: it shows the strength of the relationship and whether classes form separate clouds.

##### 2. What is/are the insight(s) found from the chart?

Brightness and edge density are strongly and positively related (r = 0.87): brighter images simply contain more edges. Classes form partly separate clouds along that diagonal - glioma at the low end, no tumour and pituitary at the high end - with substantial overlap in the middle (meningioma spreads across the range).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The overlap in the middle is where a model has to use *real* image content, and that is where errors will concentrate. For the business case this suggests looking at confusion between meningioma and its neighbours first. The strong correlation also means these hand-made variables are redundant with one another - not an issue for a CNN, but it explains why I keep them out of the final model.

#### Chart - 11 - Left-right symmetry

In [ ]:
# Chart - 11 visualization code
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(data=feat, x="class_name", y="lr_symmetry", order=CLASS_NAMES, hue="class_name",
               palette=PAL, inner="quartile", legend=False, ax=ax, cut=0)
ax.set_xticklabels(LABELS); ax.set_xlabel(""); ax.set_ylabel("Correlation with mirrored image")
ax.set_title("Left-right symmetry by class")
savefig("11_symmetry_by_class")

##### 1. Why did you pick the specific chart?

The hypothesis that a large mass breaks the near-symmetry of a healthy axial slice is clinically intuitive; a violin plot tests it visually per class.

##### 2. What is/are the insight(s) found from the chart?

The expected pattern does *not* appear: no-tumour images are not the most symmetric. Glioma has the highest median symmetry (0.78) and pituitary the lowest (0.63), but pituitary slices are mostly sagittal views, which are not left-right symmetric by anatomy. Symmetry mainly reflects viewing plane. Its effect size is small compared with brightness (Kruskal-Wallis epsilon-squared 0.11 vs 0.58).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

A useful negative result: a tempting hand-crafted 'tumours break symmetry' feature would mostly encode the imaging plane, so it would not generalise. This supports a data-driven CNN and warns against adding symmetry-based rules to the app. Hand-crafted features here help understand the data, not diagnose.

#### Chart - 12 - JPEG file size

In [ ]:
# Chart - 12 visualization code
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=feat, x="class_name", y="file_kb", order=CLASS_NAMES, hue="class_name",
            palette=PAL, legend=False, ax=ax, fliersize=2)
ax.set_xticklabels(LABELS); ax.set_xlabel(""); ax.set_ylabel("File size (KB)")
ax.set_title(f"JPEG size by class (correlation with edge density: r = {feat.file_kb.corr(feat.edge_density):.2f})")
savefig("12_file_size_by_class")

##### 1. Why did you pick the specific chart?

File size is a free, model-independent proxy for image detail (compression grows with detail); comparing it across classes checks whether the files themselves differ in a way a model could pick up.

##### 2. What is/are the insight(s) found from the chart?

Median size goes from 23 KB (glioma) to 28 KB (meningioma), 36 KB (pituitary) and 39 KB (no tumour), and tracks edge density closely (r = 0.88). Classes were therefore saved with different amount of detail/noise.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This is a data-provenance flag, not a modelling input: a model that sees raw pixels can in principle detect such compression and noise patterns. Re-encoding all images identically or adding noise/blur augmentation would reduce that risk; I do not re-encode (it would degrade the diagnostic content) but I mention it as a limitation and recommend testing on images from another source.

#### Chart - 13 - Class weights

In [ ]:
# Chart - 13 visualization code
train_labels = feat[feat.split == "train"].label.values
weights = get_class_weights(train_labels)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(LABELS, [weights[i] for i in range(4)], color=[PAL[c] for c in CLASS_NAMES])
ax.axhline(1, color="grey", ls="--", lw=1)
for i in range(4):
    ax.text(i, weights[i] + 0.02, f"{weights[i]:.2f}", ha="center")
ax.set_ylabel("Class weight"); ax.set_title("Balanced class weights (official training split)")
savefig("13_class_weights")

##### 1. Why did you pick the specific chart?

Showing the actual weights makes the imbalance treatment transparent: bars below the dashed line of 1.0 are down-weighted, bars above it are up-weighted.

##### 2. What is/are the insight(s) found from the chart?

Weights range from about 0.75 (glioma) to about 1.27 (no tumour): each no-tumour image counts about 1.7 times as much as a glioma image in the loss.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: the smaller classes - particularly 'no tumour', where an error means telling a patient a tumour exists or missing one - get a fair influence on training without duplicating images (which would add yet more leakage-like near-copies). The weights are recomputed from whichever split protocol is used.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
corr = feat[FEATURE_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation of image-level variables")
savefig("14_correlation_heatmap")

##### 1. Why did you pick the specific chart?

A correlation heatmap is the standard multivariate overview: it shows every pairwise relationship at once and reveals redundant variables.

##### 2. What is/are the insight(s) found from the chart?

Seven pairs have |r| above 0.8 - for example edge density with file size (0.88), brightness with edge density (0.87), and entropy with foreground share (0.86). Brightness, entropy, edge density, head area and file size essentially measure one thing: 'how much bright tissue and detail the image contains'. Left-right symmetry is the most independent variable (|r| up to 0.57).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Because most variables are redundant, the hand-crafted set adds little beyond a single 'amount of signal' axis, which is one more reason to let CNNs learn features from pixels. It also confirms that the large class differences found earlier are one phenomenon seen from several angles, not several independent clues.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
cols = ["brightness", "contrast", "foreground_frac", "edge_density", "lr_symmetry"]
grid = sns.pairplot(feat.sample(700, random_state=0), vars=cols, hue="class_name", hue_order=CLASS_NAMES,
                    palette=PAL, diag_kind="kde", plot_kws=dict(s=10, alpha=0.55), corner=True)
grid.figure.suptitle("Pair plot of image-level variables (700 random images)", y=1.02)
savefig("15_pair_plot")

##### 1. Why did you pick the specific chart?

A pair plot combines every scatter and every distribution in one view, which is the quickest way to see whether any combination of variables separates the classes.

##### 2. What is/are the insight(s) found from the chart?

Glioma separates well from the other classes on brightness, contrast and head-area, but meningioma, pituitary and no tumour overlap heavily on most pairs; no single pair separates all four classes. The diagonal densities show glioma peaked and narrow, no tumour wide.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive: even a flexible model on these few variables cannot fully separate all classes, so there is real room - and need - for a deep model that reads image structure. Negative: the clean separation of glioma from the rest on trivial variables again points to acquisition bias for that class.

#### Chart - 16 - PCA of raw pixels

In [ ]:
# Chart - 16 visualization code
X32 = np.stack([load_gray(p, 32).ravel() for p in feat.path])
pca = PCA(n_components=2, random_state=0).fit(X32)
coords = pca.transform(X32)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for cls in CLASS_NAMES:
    m = (feat.class_name == cls).values
    ax.scatter(coords[m, 0], coords[m, 1], s=9, alpha=0.5, color=PAL[cls], label=DISPLAY_NAMES[cls])
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})"); ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
ax.legend(title="Class"); ax.set_title("PCA of 32x32 pixel intensities - all images")
savefig("16_pca_pixels")

##### 1. Why did you pick the specific chart?

PCA squeezes 1,024 pixel values per image into two axes, giving a multivariate picture of how the raw images cluster without using the labels.

##### 2. What is/are the insight(s) found from the chart?

The first two components explain about 49% of the variance (38% + 10%). Without ever seeing the labels, raw pixels already arrange the classes: glioma forms a compact cluster at the left edge, pituitary sits lower right, no tumour is widely scattered, and meningioma overlaps all of them. Only glioma is relatively isolated; the rest overlap heavily, and it takes about 83 components to retain 90% of the variance.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The high number of components needed says the images are genuinely complex, so a linear projection is not enough and a deep network is warranted. It also means PCA would be a poor preprocessing step for the CNNs (it is used here only for visualisation).

#### Chart - 17 - Shortcut check

In [ ]:
# Chart - 17 visualization code
# How well can a model classify using ONLY seven global statistics (no spatial information)?
# Train on the official train split, test on the official test split, then separate the
# test images into clean ones and the leaked ones.
SHORTCUT_COLS = ["brightness", "contrast", "skewness", "entropy", "foreground_frac", "edge_density", "lr_symmetry"]
tr, te = feat[feat.split == "train"], feat[feat.split == "test"]
rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(tr[SHORTCUT_COLS], tr.label)
shortcut_correct = (rf.predict(te[SHORTCUT_COLS]) == te.label).values
shortcut_clean = te.leak_free.values
scores = {
    "Always guess\nlargest class": te.label.value_counts(normalize=True).max(),
    "Official test\n(all %d)" % len(te): shortcut_correct.mean(),
    "Clean test\n(%d)" % shortcut_clean.sum(): shortcut_correct[shortcut_clean].mean(),
    "Leaked test images\n(%d)" % (~shortcut_clean).sum(): shortcut_correct[~shortcut_clean].mean(),
}
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(list(scores), [v * 100 for v in scores.values()], color=["#999999", "#4c78a8", "#4c9a2a", "#c0392b"])
for b, v in zip(bars, scores.values()):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 1, f"{v:.1%}", ha="center")
ax.set_ylim(0, 112); ax.set_ylabel("Accuracy (%)")
ax.set_title("Random forest on 7 global image statistics (no spatial information)")
savefig("17_shortcut_check")

##### 1. Why did you pick the specific chart?

A deliberately naive model is the cleanest experiment for a suspicion of dataset bias: if a model that cannot see anatomy still scores well, the data contain shortcuts. Splitting the result by leaked/clean test images isolates the effect of the leakage.

##### 2. What is/are the insight(s) found from the chart?

Seven numbers per image - nothing about shapes or locations - give about 87% accuracy on the official test set versus 33% for guessing, and about 83% on the clean test images. On the 67 leaked test images the same crude model is right **every time (100%)**. Leaked images are near-copies of training images, so even a crude model finds their twin.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Two warnings with direct business impact. (1) Part of any high accuracy on this dataset is available without recognising tumours, so deep-model accuracy must be compared with this floor and validated on external data. (2) Leakage inflates results, so I evaluate on a grouped, leak-free split. Positive: because I measured both effects, the final report can state what the models genuinely add beyond these shortcuts.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

1. **Brightness differs between the four classes** (Charts 5-7): this decides how seriously to take the acquisition-shortcut risk.
2. **The class mix is the same in train, valid and test** (Chart 2): this checks that the official split was stratified, i.e. that test scores are comparable.
3. **A model without spatial information is more accurate on leaked test images than on clean ones** (Chart 17): this tests whether the leakage genuinely inflates scores or is just a coincidence of small numbers.

All tests use a significance level of 0.05.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** the distribution of mean image brightness is the same in all four classes.  
**H1:** at least one class has a different brightness distribution.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Hypothesis 1)
groups = [feat.loc[feat.class_name == c, "brightness"] for c in CLASS_NAMES]
H, p1 = stats.kruskal(*groups)
epsilon_sq = (H - len(groups) + 1) / (len(feat) - len(groups))     # effect size
print(f"Kruskal-Wallis H = {H:.1f}, p-value = {p1:.3g}, epsilon-squared = {epsilon_sq:.2f}")
# Which pairs differ? Pairwise Mann-Whitney tests with Bonferroni correction (6 pairs)
from itertools import combinations
for a, b in combinations(CLASS_NAMES, 2):
    u, p = stats.mannwhitneyu(feat.loc[feat.class_name == a, "brightness"],
                              feat.loc[feat.class_name == b, "brightness"])
    print(f"  {a:10s} vs {b:10s} adjusted p = {min(p * 6, 1):.2g}")
print("Reject H0" if p1 < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

A **Kruskal-Wallis H test** (with Bonferroni-corrected pairwise Mann-Whitney U tests as follow-up).

##### Why did you choose the specific statistical test?

Brightness is numeric and the groups are four independent samples, which suggests one-way ANOVA - but the violin plots show skewed distributions with very different spreads, which violates ANOVA's assumptions. Kruskal-Wallis is its rank-based counterpart and makes no normality or equal-variance assumption. **Result:** p is far below 0.05 (effect size epsilon-squared about 0.58, large), so we reject H0: brightness differs by class, and every pair of classes differs significantly. Brightness - an acquisition property - carries strong class information.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** split (train/valid/test) and class are independent - each split has the same class mix.  
**H1:** the class mix depends on the split.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Hypothesis 2)
table = pd.crosstab(feat.split, feat.class_name)[CLASS_NAMES]
chi2, p2, dof, expected = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print(f"chi-square = {chi2:.2f}, dof = {dof}, p-value = {p2:.3f}, Cramer's V = {cramers_v:.3f}")
print("Smallest expected count:", expected.min().round(1), "(rule of thumb: should be >= 5)")
print("Reject H0" if p2 < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

A **chi-square test of independence** on the 3x4 split-by-class contingency table.

##### Why did you choose the specific statistical test?

Both variables are categorical and we compare observed counts with the counts expected if the split did not matter - exactly what the chi-square test of independence does. All expected counts are well above 5, so its approximation is valid. **Result:** p is about 0.47, so we fail to reject H0 (Cramer's V about 0.03, negligible): the official split is stratified by class. Note this says nothing about *scan-level* leakage, which this test cannot see.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** a model that uses only global image statistics is equally likely to classify a leaked test image and a clean test image correctly.  
**H1:** it is more likely to classify leaked test images correctly (leakage inflates scores).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Hypothesis 3)
# 2x2 table: rows = leaked / clean test images, columns = correct / wrong
leaked_ok, leaked_bad = int(shortcut_correct[~shortcut_clean].sum()), int((~shortcut_correct[~shortcut_clean]).sum())
clean_ok, clean_bad = int(shortcut_correct[shortcut_clean].sum()), int((~shortcut_correct[shortcut_clean]).sum())
contingency = [[leaked_ok, leaked_bad], [clean_ok, clean_bad]]
print(pd.DataFrame(contingency, index=["leaked", "clean"], columns=["correct", "wrong"]))
odds, p3 = stats.fisher_exact(contingency, alternative="greater")
print(f"Fisher exact test (one-sided): p-value = {p3:.2g}")
print("Reject H0" if p3 < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

A one-sided **Fisher's exact test** on the 2x2 table (leaked/clean x correct/wrong).

##### Why did you choose the specific statistical test?

We compare two proportions of successes in two independent groups, and one cell is zero (no leaked image was misclassified), which makes the chi-square approximation unreliable; Fisher's exact test is exact for small counts. The alternative is one-sided because the question is specifically whether leakage *inflates* accuracy. **Result:** p is well below 0.001, so we reject H0: leaked images are classified correctly significantly more often, even by a model that cannot see anatomy. That justifies evaluating on a leak-free split.

## ***6. Feature Engineering & Data Pre-processing***

*This is an image-classification project, so several template steps (text, categorical columns, tabular imputation) do not apply; each sub-section states what was done or why it is not needed.*

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
print("Null metadata cells:", int(df_official.isna().sum().sum()), "| unreadable images:", n_unreadable)

#### What all missing value imputation techniques have you used and why did you use those techniques?

None. There are no missing metadata values and no unreadable images (Section 1), so there is nothing to impute or remove. Imputing pixels would be inappropriate for medical images anyway.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments: count IQR outliers in brightness per class
rows = []
for cls in CLASS_NAMES:
    s = feat.loc[feat.class_name == cls, "brightness"]
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    rows.append({"class": cls, "images": len(s),
                 "outliers (1.5 x IQR)": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())})
outliers = pd.DataFrame(rows).set_index("class")
outliers["% of class"] = (100 * outliers["outliers (1.5 x IQR)"] / outliers["images"]).round(1)
display(outliers)

# Look at the five brightest images: are they errors or legitimate scans?
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (_, row) in zip(axes, feat.nlargest(5, "brightness").iterrows()):
    ax.imshow(Image.open(row.path).convert("RGB")); ax.axis("off"); ax.set_title(row.class_name, fontsize=9)
plt.tight_layout(); plt.show()

##### What all outlier treatment techniques have you used and why did you use those techniques?

None were removed. Meningioma has the most statistical outliers (the very bright contrast-enhanced masses, about 13% of the class) and the brightest images shown above are valid scans - bright masses and different MRI sequences, not corrupt data. Deleting them would remove exactly the cases the model must learn, so instead robustness is added through brightness/contrast augmentation.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns: the target is stored as an integer label per class
label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
print(label_map)
assert (df_official.label == df_official.class_name.map(label_map)).all()

#### What all categorical encoding techniques have you used & why did you use those techniques?

**Integer (label) encoding** of the four classes, trained with *sparse categorical cross-entropy*. The classes are mutually exclusive and unordered; the loss treats integers as class ids (not magnitudes), so no one-hot matrix is needed, and the model's softmax output is indexed by the same ids. The folder name is the single source of truth (it agrees with the CSV labels for every image).

### 4. Textual Data Preprocessing
*Not applicable - the dataset contains no text. (Template steps 1-10 skipped.)*

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Which of the hand-crafted variables are redundant? (|r| > 0.8)
corr = feat[FEATURE_COLUMNS].corr()
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURE_COLUMNS) for b in FEATURE_COLUMNS[i + 1:] if abs(corr.loc[a, b]) > 0.8]
pd.DataFrame(pairs, columns=["variable A", "variable B", "r"])

#### 2. Feature Selection

##### What all feature selection methods have you used  and why?
No feature selection. The deep models take raw pixels and learn their own features end to end; the hand-crafted variables above were only used to *understand* the data (and for the shortcut check and baseline). Seven pairs of them are strongly redundant (|r| > 0.8), so selecting among them would add nothing.

##### Which all features you found important and why?
For the CNNs, importance is not a column list but image regions - shown with occlusion-sensitivity maps in the explainability step of Section 7. For the data audit, brightness-type variables carry the most class information (Section 5, Hypothesis 1).

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data: resize 640x640 -> 224x224 (anti-aliased), cast to float, scale to [0, 1]
sample_path = df_official.path.iloc[0]
raw = Image.open(sample_path)
arr = preprocess_pil(raw)
print(f"raw: {raw.size} {raw.mode} uint8 0-255   ->   model input: {arr.shape} {arr.dtype}, range [{arr.min():.2f}, {arr.max():.2f}]")
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(raw); axes[0].set_title("raw 640x640"); axes[1].imshow(arr); axes[1].set_title("model input 224x224")
for a in axes: a.axis("off")
plt.show()

Yes: (1) **resize to 224x224**, the input size the ImageNet backbones were trained for, with an *anti-aliased* bilinear filter - the same filter is used in the training pipeline and in the Streamlit app so training and serving see identical pixels; no cropping, so tumours near the edge are kept; (2) **grayscale MRI stored as 3-channel RGB is kept as 3 channels**, because pretrained backbones expect three; (3) **augmentation** (Section 7.0) as a training-only transformation.

### 6. Data Scaling

##### Which method have you used to scale you data and why?

Pixel values are divided by 255, giving the **[0, 1] range** required by the brief for every model. Each transfer model then converts [0, 1] to what its backbone was pre-trained on with a `Rescaling` layer *inside the model* (to [-1, 1] for MobileNetV2/ResNet50V2, to [0, 255] for EfficientNetB0), so the saved `.h5` files all accept the same input and the app needs no per-model preprocessing. Per-image standardisation was not used because it would erase absolute intensity, which carries diagnostic meaning.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Not for modelling: convolutional layers exploit the spatial structure of the full image, and a flattening/PCA step would destroy it. Global average pooling inside the networks already reduces each feature map to one number.

In [ ]:
# DImensionality Reduction (visualisation only): how many PCA components keep 80/90/95% of variance?
X32 = np.stack([load_gray(p, 32).ravel() for p in feat.path])
cum = PCA(random_state=0).fit(X32).explained_variance_ratio_.cumsum()
print({f"{int(t*100)}% variance": int(np.searchsorted(cum, t)) + 1 for t in (0.8, 0.9, 0.95)}, "components out of", X32.shape[1])

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

**PCA, for visualisation only** (Chart 16). It needs about 83 components for 90% of the variance, so a linear reduction would be lossy; the models are trained on full images.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
df = load_dataframe(DATA_DIR, SPLIT_MODE)

train_df = df[df.split == "train"]
val_df = df[df.split == "valid"]
if SPLIT_MODE == "official":
    val_df = val_df[val_df.leak_free]          # keep early stopping honest: drop leaked validation images
test_df = df[df.split == "test"]

print(f"SPLIT_MODE = {SPLIT_MODE!r}:  train {len(train_df)}  |  validation {len(val_df)}  |  test {len(test_df)}"
      f"  ({len(test_df) / len(df):.0%} test)")
display(pd.crosstab(df.split, df.class_name)[CLASS_NAMES].loc[["train", "valid", "test"]])

# Proof that no original scan is shared between splits (only guaranteed in 'grouped' mode)
ids = {s: set(df.loc[df.split == s, "source_id"]) for s in ("train", "valid", "test")}
print("scans shared  train/valid:", len(ids["train"] & ids["valid"]),
      " train/test:", len(ids["train"] & ids["test"]), " valid/test:", len(ids["valid"] & ids["test"]))
print("test images flagged as leaked:", int((~test_df.leak_free).sum()))

##### What data splitting ratio have you used and why?

**Default (`grouped`): about 71% train / 14% validation / 14% test (1,745 / 349 / 349 images)**, stratified by class and grouped by original scan (`StratifiedGroupKFold`), so the splits share no scan and keep the class proportions (33/22/20/25%). 14% each leaves a test set of 349 images (115 gliomas) that is large enough for stable per-class estimates; the validation set is used only for early stopping, learning-rate decay and model selection, never for the final report.

**Why not the official split?** It leaks (Section 1: 164 validation/test images are variants of other-split scans, all gliomas) and its leak-free test set contains only 13 gliomas - too few to say anything reliable about that class. `SPLIT_MODE = "official"` is still supported; it reports results on the full official test set *and* on its leak-free subset.

**Remaining limitation:** the dataset has no patient id, so different slices from the same patient can still fall in different splits. True clinical generalisation needs an external, patient-level test set.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Mildly: the largest class (glioma, 33%) is about 1.7x the smallest (no tumour, 20%). That is not severe enough for resampling, but it is enough to bias a model toward glioma if the loss ignores it.

In [ ]:
# Handling Imbalanced Dataset (If needed): balanced class weights from the TRAINING split only
class_weights = get_class_weights(train_df.label.values)
pd.DataFrame({"class": CLASS_NAMES, "train images": train_df.label.value_counts().sort_index().values,
              "class weight": [round(class_weights[i], 3) for i in range(4)]}).set_index("class")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

**Class weights** (`n / (4 x class count)`) passed to `model.fit`, so errors on smaller classes cost more in the loss. Oversampling or duplicating images was avoided because duplicates of one scan across train and validation are exactly the leakage found in Section 1. Evaluation uses macro-averaged precision/recall/F1 and per-class reports so a weak small class cannot hide behind overall accuracy.

## ***7. ML Model Implementation***

### 7.0 Setup: TensorFlow, augmentation, input pipelines

In [ ]:
# TensorFlow is only needed from here on.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU") or "none (training will be slow - enable a GPU runtime)")

# Training budget
if FAST_MODE:
    EPOCHS_CNN, HEAD_EPOCHS, FT_EPOCHS, TUNE_TRIALS, TUNE_EPOCHS = 2, 1, 1, 1, 1
else:
    EPOCHS_CNN, HEAD_EPOCHS, FT_EPOCHS, TUNE_TRIALS, TUNE_EPOCHS = 40, 12, 15, 4, 12

The model-building and training functions below are verbatim copies of `src/models.py` and `src/train.py`.

In [ ]:
# src/models.py

def build_augmentation() -> keras.Sequential:
    """Training-time augmentation (rotation, flips, zoom, shifts, brightness, contrast).

    Fill mode is 'constant' (black) because MRI backgrounds are black; the default
    'reflect' would mirror anatomy into the background and create unrealistic images.
    """
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=SEED),
            layers.RandomRotation(0.06, fill_mode="constant", seed=SEED),      # about +-22 deg
            layers.RandomZoom(0.15, fill_mode="constant", seed=SEED),
            layers.RandomTranslation(0.08, 0.08, fill_mode="constant", seed=SEED),
            layers.RandomBrightness(0.15, value_range=(0.0, 1.0), seed=SEED),
            layers.RandomContrast(0.15, seed=SEED),
        ],
        name="augmentation",
    )


def build_custom_cnn(num_classes: int = 4, base_filters: int = 32,
                     dropout: float = 0.4, input_shape=INPUT_SHAPE) -> keras.Model:
    """4 blocks of (Conv-BN-ReLU) x2 + MaxPool + Dropout, then GAP + dense head.

    BatchNorm stabilises learning; Dropout and global average pooling (instead of a huge
    Flatten->Dense) limit over-fitting on a ~1.7k-image training set.
    """
    inputs = keras.Input(shape=input_shape, name="image")
    x = inputs
    for block, mult in enumerate([1, 2, 4, 8], start=1):
        for conv in (1, 2):
            x = layers.Conv2D(base_filters * mult, 3, padding="same", use_bias=False,
                              name=f"conv{block}_{conv}")(x)
            x = layers.BatchNormalization(name=f"bn{block}_{conv}")(x)
            x = layers.Activation("relu", name=f"relu{block}_{conv}")(x)
        x = layers.MaxPooling2D(name=f"pool{block}")(x)
        x = layers.Dropout(dropout * 0.5, name=f"drop{block}")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, use_bias=False, name="dense")(x)
    x = layers.BatchNormalization(name="dense_bn")(x)
    x = layers.Activation("relu", name="dense_relu")(x)
    x = layers.Dropout(dropout, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probs")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def build_transfer_model(name: str, num_classes: int = 4, dropout: float = 0.3,
                         input_shape=INPUT_SHAPE) -> keras.Model:
    """ImageNet backbone (frozen) + new classification head."""
    if name not in TRANSFER_BACKBONES:
        raise ValueError(f"Unknown backbone '{name}'. Choose from {list(TRANSFER_BACKBONES)}")
    class_name, rescale_mode = TRANSFER_BACKBONES[name]
    backbone_cls = getattr(keras.applications, class_name)

    inputs = keras.Input(shape=input_shape, name="image")
    if rescale_mode == "minus1_1":
        x = layers.Rescaling(2.0, offset=-1.0, name="to_minus1_1")(inputs)
    else:
        x = layers.Rescaling(255.0, name="to_0_255")(inputs)

    backbone = backbone_cls(include_top=False, weights="imagenet", input_shape=input_shape)
    backbone.trainable = False                     # stage 1: train the new head only
    x = backbone(x, training=False)                # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="drop_1")(x)
    x = layers.Dense(256, activation="relu", name="dense")(x)
    x = layers.Dropout(dropout, name="drop_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probs")(x)
    return keras.Model(inputs, outputs, name=name)


def get_backbone(model: keras.Model) -> keras.Model:
    """Return the nested pre-trained backbone of a transfer model."""
    for layer in model.layers:
        if isinstance(layer, keras.Model):
            return layer
    raise ValueError("No nested backbone found - is this a transfer model?")


def unfreeze_top_layers(model: keras.Model, n_layers: int = 40) -> int:
    """Stage 2: make the last `n_layers` of the backbone trainable.

    BatchNorm layers stay frozen: with a small dataset, updating their running
    statistics tends to hurt more than it helps. Call compile() again afterwards.
    Returns the number of trainable backbone layers.
    """
    backbone = get_backbone(model)
    backbone.trainable = True
    for layer in backbone.layers[:-n_layers]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    return sum(layer.trainable for layer in backbone.layers)


def compile_model(model: keras.Model, learning_rate: float) -> keras.Model:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_model(name: str, **kwargs) -> keras.Model:
    """Single entry point used by the training script and the notebook."""
    if name == "custom_cnn":
        return build_custom_cnn(**kwargs)
    return build_transfer_model(name, **kwargs)

In [ ]:
# src/train.py

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def make_callbacks(weights_path: Path, patience: int, monitor: str = "val_loss"):
    """EarlyStopping + ModelCheckpoint (best weights) + ReduceLROnPlateau.

    Checkpoints store weights only, in '<name>.weights.h5' - the one file name that is
    valid in both Keras 2 and Keras 3. The full model is saved to <name>.h5 afterwards.
    """
    return [
        keras.callbacks.EarlyStopping(monitor=monitor, patience=patience,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(str(weights_path), monitor=monitor,
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor=monitor, factor=0.5,
                                          patience=max(patience // 2, 1),
                                          min_lr=1e-7, verbose=1),
    ]


def fit_stage(model, train_ds, val_ds, class_weights, epochs, learning_rate,
              weights_path: Path, patience: int, verbose: int = 2) -> dict:
    """Compile and fit one training stage; returns the history dict."""
    compile_model(model, learning_rate)
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        class_weight=class_weights,
                        callbacks=make_callbacks(weights_path, patience),
                        verbose=verbose)
    return history.history


def train_model(name: str, train_ds, val_ds, class_weights: dict,
                models_dir: Path = MODELS_DIR, epochs: int = 40,
                head_epochs: int = 12, finetune_epochs: int = 15,
                finetune_layers: int = 40, learning_rate: float = 1e-3,
                build_kwargs: dict | None = None, verbose: int = 2):
    """Train one model end to end and save it as models/<name>.h5.

    custom_cnn : single stage, `epochs` max (early stopping decides when to stop)
    transfer   : stage 1 trains the new head (lr 1e-3); stage 2 unfreezes the top
                 `finetune_layers` backbone layers and continues at lr 1e-5
    Returns (model, history, finetune_start_epoch or None).
    """
    models_dir = Path(models_dir)
    models_dir.mkdir(parents=True, exist_ok=True)
    set_seed()
    model = build_model(name, **(build_kwargs or {}))
    weights_path = models_dir / f"{name}.weights.h5"

    if name == "custom_cnn":
        history = fit_stage(model, train_ds, val_ds, class_weights, epochs,
                            learning_rate, weights_path, patience=8, verbose=verbose)
        finetune_start = None
    else:
        history = fit_stage(model, train_ds, val_ds, class_weights, head_epochs,
                            learning_rate, weights_path, patience=4, verbose=verbose)
        finetune_start = len(history["loss"])
        n_trainable = unfreeze_top_layers(model, finetune_layers)
        print(f"[{name}] fine-tuning {n_trainable} backbone layers")
        stage2 = fit_stage(model, train_ds, val_ds, class_weights, finetune_epochs,
                           1e-5, weights_path, patience=5, verbose=verbose)
        history = {k: history[k] + stage2[k] for k in history}

    # EarlyStopping(restore_best_weights=True) already restored the best weights of the
    # last stage; save the complete model in the .h5 format requested by the brief.
    model.save(str(models_dir / f"{name}.h5"))
    return model, history, finetune_start


def random_search_cnn(train_ds, val_ds, class_weights, n_trials: int = 4,
                      epochs: int = 12, verbose: int = 0) -> tuple[pd.DataFrame, dict]:
    """Small random search over custom-CNN hyper-parameters, scored on validation loss.

    k-fold cross-validation would multiply training time by k; with ~1.7k images and a
    fixed validation split a short random search is the practical alternative.
    """
    space = {"learning_rate": [1e-3, 5e-4, 2e-4],
             "dropout": [0.3, 0.4, 0.5],
             "base_filters": [16, 32]}
    rng = random.Random(SEED)
    rows = []
    for trial in range(n_trials):
        params = {k: rng.choice(v) for k, v in space.items()}
        set_seed(SEED + trial)
        model = build_model("custom_cnn", base_filters=params["base_filters"],
                            dropout=params["dropout"])
        compile_model(model, params["learning_rate"])
        hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                         class_weight=class_weights, verbose=verbose,
                         callbacks=[keras.callbacks.EarlyStopping(
                             monitor="val_loss", patience=4, restore_best_weights=True)]
                         ).history
        best = int(np.argmin(hist["val_loss"]))
        rows.append({**params, "val_loss": hist["val_loss"][best],
                     "val_accuracy": hist["val_accuracy"][best], "epochs_run": len(hist["loss"])})
        print(f"trial {trial + 1}/{n_trials}: {rows[-1]}")
    table = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    best_params = {k: table.loc[0, k] for k in space}
    best_params["base_filters"] = int(best_params["base_filters"])
    return table, best_params

In [ ]:
# Input pipelines (decode -> resize -> [0,1] -> cache; augmentation on the training set only)
set_seed()
augmentation = build_augmentation()
train_ds = make_dataset(train_df.path, train_df.label, True, BATCH_SIZE, augmentation=augmentation)
val_ds = make_dataset(val_df.path, val_df.label, False, BATCH_SIZE)
test_ds = make_dataset(test_df.path, test_df.label, False, BATCH_SIZE)

# Look at what augmentation does to a batch
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, img, lab in zip(axes.ravel(), images.numpy(), labels.numpy()):
    ax.imshow(img); ax.set_title(DISPLAY_NAMES[CLASS_NAMES[lab]], fontsize=9); ax.axis("off")
plt.suptitle("Augmented training batch (rotation, flips, zoom, shifts, brightness, contrast)")
plt.tight_layout(); plt.show()

**Augmentation choices.** Rotation (about +-22 degrees), horizontal and vertical flips, zoom (15%), shifts (8%), brightness and contrast jitter (15%) - as listed in the brief. Empty areas are filled with **black**, not mirrored, because MRI backgrounds are black and reflected anatomy would be unrealistic. Augmentation is applied to batches *after* caching so each epoch sees new variants, and never to validation or test images.

### ML Model - 0 (reference): classical baseline - HOG features + SVM

Before any deep model, a classical pipeline sets the bar the CNNs must clear: **Histogram of Oriented Gradients** descriptors (local edge-orientation histograms on a 96x96 grayscale image) feed an **RBF support-vector machine**. Unlike the 7-number shortcut model, HOG *does* encode spatial structure. Hyper-parameters are tuned with a grid search and **group-aware 3-fold cross-validation** (`GroupKFold` on the original scan id), because plain k-fold would put variants of the same scan on both sides of a fold and over-estimate accuracy. It needs no GPU.

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, GroupKFold

def hog_features(paths, size=96):
    """HOG descriptor of the grayscale image resized to size x size."""
    return np.stack([hog(load_gray(p, size), orientations=9, pixels_per_cell=(12, 12),
                         cells_per_block=(2, 2)) for p in paths])

t0 = time.time()
X_hog = hog_features(df.path)                       # rows follow df.index
print("HOG matrix", X_hog.shape, f"({time.time() - t0:.0f}s)")

search = GridSearchCV(SVC(kernel="rbf"), {"C": [1, 10, 100], "gamma": ["scale", 1e-3]},
                      cv=GroupKFold(n_splits=3), n_jobs=-1)
search.fit(X_hog[train_df.index], train_df.label, groups=train_df.source_id)
print("best parameters:", search.best_params_, "| grouped-CV accuracy:", round(search.best_score_, 3))

# Evaluate on the test split (and on its leak-free subset, which differs only in 'official' mode)
scores = search.decision_function(X_hog[test_df.index])
mask = test_df.leak_free.to_numpy()
baseline = {"all": evaluate_predictions(test_df.label.values, scores, CLASS_NAMES),
            "leak_free": evaluate_predictions(test_df.label.values[mask], scores[mask], CLASS_NAMES)}
print(f"Baseline - accuracy {baseline['leak_free']['accuracy']:.3f} (95% CI {baseline['leak_free']['accuracy_ci'][0]:.3f}-"
      f"{baseline['leak_free']['accuracy_ci'][1]:.3f}), macro-F1 {baseline['leak_free']['macro_f1']:.3f}  [n = {baseline['leak_free']['n']}]")
if SPLIT_MODE == "official":
    print(f"On the full official test set (with leaked images): accuracy {baseline['all']['accuracy']:.3f}")
plot_confusion_matrix(baseline["leak_free"]["confusion_matrix"], CLASS_NAMES, "HOG + SVM baseline")
plt.show()

**Reading the baseline.** This is a real, trained model, and it is the reference for the deep models: a CNN that cannot beat HOG + SVM is not worth its cost. Its confusion matrix also shows which classes are intrinsically hard to separate with edge features alone.

### Experiment runner

`run_experiment` trains one model, saves `models/<name>.h5`, evaluates it on the test split (and its leak-free subset), and draws the training curves, confusion matrix and per-class metric chart. Results are collected in `RESULTS` for the comparison in Section 7.6.

In [ ]:
RESULTS, HISTORIES = {}, {}

def run_experiment(name, **train_kwargs):
    """Train -> save .h5 -> evaluate -> plot. Returns the trained model."""
    model, history, finetune_start = train_model(name, train_ds, val_ds, class_weights, MODELS_DIR,
                                                 verbose=2, **train_kwargs)
    HISTORIES[name] = (history, finetune_start)
    json.dump({"history": {k: [float(v) for v in vs] for k, vs in history.items()},
               "finetune_start": finetune_start}, open(REPORTS_DIR / f"history_{name}.json", "w"))

    y_true, y_prob = predict_probs(model, test_ds)
    clean = test_df.leak_free.to_numpy()
    path = MODELS_DIR / f"{name}.h5"
    RESULTS[name] = {"all": evaluate_predictions(y_true, y_prob, CLASS_NAMES),
                     "leak_free": evaluate_predictions(y_true[clean], y_prob[clean], CLASS_NAMES),
                     "params": model.count_params(), "size_mb": path.stat().st_size / 1e6,
                     "latency_ms": measure_latency_ms(model)}
    r = RESULTS[name]["leak_free"]
    print(f"\n{name}: test accuracy {r['accuracy']:.3f}  macro-F1 {r['macro_f1']:.3f}  "
          f"(95% CI for accuracy {r['accuracy_ci'][0]:.3f}-{r['accuracy_ci'][1]:.3f}, n = {r['n']})")

    # --- figures: training curves | confusion matrix | per-class metrics
    plot_history(history, name, finetune_start); plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    plot_confusion_matrix(r["confusion_matrix"], CLASS_NAMES, f"{name} - confusion matrix", ax=axes[0])
    per_class = pd.DataFrame(r["per_class"]).T[["precision", "recall", "f1"]]
    per_class.plot(kind="bar", ax=axes[1], rot=20, ylim=(0, 1.05))
    axes[1].set_title(f"{name} - per-class metrics"); axes[1].legend(loc="lower right")
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(r["per_class"]).T.round(3))
    return model

### ML Model - 1: Custom CNN

A convolutional network built from scratch: **four blocks** of *two 3x3 convolutions, each followed by batch normalisation and ReLU*, then max-pooling and dropout; filter counts double per block (32-64-128-256). **Global average pooling** replaces a large flatten layer, followed by a 256-unit dense layer with batch norm and dropout and a 4-way softmax. Batch normalisation stabilises and speeds up learning; dropout and global pooling limit over-fitting on a small (about 1.7k image) training set. About 1.2 million parameters - small enough to train quickly and run on a CPU.

The learning rate, dropout and width are chosen first by a small random search (below); the final model is then trained with the best settings.

#### 2. Cross-validation & Hyperparameter Tuning (done first, because the final model uses the result)

In [ ]:
# Random search: learning rate x dropout x base filters, scored on validation loss
tune_table, best_params = random_search_cnn(train_ds, val_ds, class_weights,
                                            n_trials=TUNE_TRIALS, epochs=TUNE_EPOCHS)
display(tune_table.round(4))
print("Best parameters:", best_params)

##### Which hyperparameter optimization technique have you used and why?

**Random search** over learning rate {1e-3, 5e-4, 2e-4}, head dropout {0.3, 0.4, 0.5} and base filters {16, 32}, scored by validation loss with early stopping per trial. Random search covers a space more efficiently than a full grid when only a few settings matter, and a short search fits a GPU budget. **k-fold cross-validation was not used for the deep models**: it would multiply training time by k, and a fixed, group-aware validation split of 349 images is the standard, adequate substitute at this scale (cross-validation *is* used for the SVM baseline).

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 1 Implementation (best settings from the search) + evaluation chart
cnn = run_experiment("custom_cnn", epochs=EPOCHS_CNN, learning_rate=float(best_params["learning_rate"]),
                     build_kwargs={"base_filters": int(best_params["base_filters"]),
                                   "dropout": float(best_params["dropout"])})

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
# Improvement attributable to tuning: best vs. median trial (validation loss / accuracy)
if len(tune_table) > 1:
    print(f"validation loss   best trial {tune_table.val_loss.min():.3f}  vs  median trial {tune_table.val_loss.median():.3f}")
    print(f"validation acc.   best trial {tune_table.val_accuracy.max():.3f}  vs  median trial {tune_table.val_accuracy.median():.3f}")
else:
    print("Only one tuning trial (FAST_MODE) - no comparison possible.")
h, _ = HISTORIES["custom_cnn"]
print(f"Final model: best validation accuracy {max(h['val_accuracy']):.3f} after {len(h['loss'])} epochs "
      f"(early stopping restored the best epoch).")

The size of the improvement is printed above for the actual run; the point of the comparison is to show whether tuning mattered. If best and median trials are close, the architecture is robust to these settings.

### ML Model - 2: MobileNetV2 (transfer learning)

**MobileNetV2** (about 2.3 M backbone parameters) uses *depthwise-separable convolutions* and *inverted residual blocks with linear bottlenecks*, designed for fast, light inference. It is the efficiency candidate: if it matches the heavier models, it is the natural choice for a web app that may run on a CPU.

**Two-stage training.** *Stage 1:* the ImageNet backbone is frozen and only the new head (global average pooling, dropout, 256-unit dense layer, dropout, 4-way softmax) is trained at learning rate 1e-3. *Stage 2:* the top 40 backbone layers are unfrozen (BatchNorm layers stay frozen) and training continues at 1e-5, so pretrained features are adapted gently to MRI without being destroyed. A dashed line in the training curves marks where stage 2 begins.

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 2 Implementation: MobileNetV2
mobilenetv2_model = run_experiment("mobilenetv2", head_epochs=HEAD_EPOCHS, finetune_epochs=FT_EPOCHS)

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# The "tuning" for this model is the fine-tuning stage: compare the end of stage 1 with the best epoch overall
h, ft = HISTORIES["mobilenetv2"]
head_best = max(h["val_accuracy"][:ft])
overall_best = max(h["val_accuracy"])
print(f"validation accuracy - head only (stage 1): {head_best:.3f} | after fine-tuning (best epoch overall): {overall_best:.3f} "
      f"| change: {overall_best - head_best:+.3f}")

##### Which hyperparameter optimization technique have you used and why?

A **two-stage schedule** rather than a search: the head is trained at 1e-3, then the top 40 layers are fine-tuned at 1e-5, which is the standard, reliable recipe for small datasets. `ReduceLROnPlateau` lowers the learning rate when validation loss stalls and `EarlyStopping` (patience 4-5, best weights restored) prevents over-fitting. A search over fine-tuning depth would multiply training time for little expected gain.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The cell above prints the validation-accuracy change from stage 1 to the best fine-tuned epoch, and the training curves show it as the step after the dashed line. A small or zero change means the frozen ImageNet features were already sufficient; a gain means adaptation to MRI texture helped.

### ML Model - 3: ResNet50V2 (transfer learning)

**ResNet50V2** (about 23.6 M backbone parameters) is a 50-layer residual network with pre-activation blocks; skip connections let gradients flow through many layers. It is the high-capacity candidate and the usual reference for medical-imaging transfer learning, at roughly ten times the size of MobileNetV2.

**Two-stage training.** *Stage 1:* the ImageNet backbone is frozen and only the new head (global average pooling, dropout, 256-unit dense layer, dropout, 4-way softmax) is trained at learning rate 1e-3. *Stage 2:* the top 40 backbone layers are unfrozen (BatchNorm layers stay frozen) and training continues at 1e-5, so pretrained features are adapted gently to MRI without being destroyed. A dashed line in the training curves marks where stage 2 begins.

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 3 Implementation: ResNet50V2
resnet50v2_model = run_experiment("resnet50v2", head_epochs=HEAD_EPOCHS, finetune_epochs=FT_EPOCHS)

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# The "tuning" for this model is the fine-tuning stage: compare the end of stage 1 with the best epoch overall
h, ft = HISTORIES["resnet50v2"]
head_best = max(h["val_accuracy"][:ft])
overall_best = max(h["val_accuracy"])
print(f"validation accuracy - head only (stage 1): {head_best:.3f} | after fine-tuning (best epoch overall): {overall_best:.3f} "
      f"| change: {overall_best - head_best:+.3f}")

##### Which hyperparameter optimization technique have you used and why?

A **two-stage schedule** rather than a search: the head is trained at 1e-3, then the top 40 layers are fine-tuned at 1e-5, which is the standard, reliable recipe for small datasets. `ReduceLROnPlateau` lowers the learning rate when validation loss stalls and `EarlyStopping` (patience 4-5, best weights restored) prevents over-fitting. A search over fine-tuning depth would multiply training time for little expected gain.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The cell above prints the validation-accuracy change from stage 1 to the best fine-tuned epoch, and the training curves show it as the step after the dashed line. A small or zero change means the frozen ImageNet features were already sufficient; a gain means adaptation to MRI texture helped.

### ML Model - 4: EfficientNetB0 (transfer learning)

**EfficientNetB0** (about 4 M backbone parameters) balances depth, width and resolution with a compound-scaling rule and squeeze-and-excitation blocks, reaching strong ImageNet accuracy at small size. It is the accuracy-per-parameter candidate.

**Two-stage training.** *Stage 1:* the ImageNet backbone is frozen and only the new head (global average pooling, dropout, 256-unit dense layer, dropout, 4-way softmax) is trained at learning rate 1e-3. *Stage 2:* the top 40 backbone layers are unfrozen (BatchNorm layers stay frozen) and training continues at 1e-5, so pretrained features are adapted gently to MRI without being destroyed. A dashed line in the training curves marks where stage 2 begins.

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 4 Implementation: EfficientNetB0
efficientnetb0_model = run_experiment("efficientnetb0", head_epochs=HEAD_EPOCHS, finetune_epochs=FT_EPOCHS)

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# The "tuning" for this model is the fine-tuning stage: compare the end of stage 1 with the best epoch overall
h, ft = HISTORIES["efficientnetb0"]
head_best = max(h["val_accuracy"][:ft])
overall_best = max(h["val_accuracy"])
print(f"validation accuracy - head only (stage 1): {head_best:.3f} | after fine-tuning (best epoch overall): {overall_best:.3f} "
      f"| change: {overall_best - head_best:+.3f}")

##### Which hyperparameter optimization technique have you used and why?

A **two-stage schedule** rather than a search: the head is trained at 1e-3, then the top 40 layers are fine-tuned at 1e-5, which is the standard, reliable recipe for small datasets. `ReduceLROnPlateau` lowers the learning rate when validation loss stalls and `EarlyStopping` (patience 4-5, best weights restored) prevents over-fitting. A search over fine-tuning depth would multiply training time for little expected gain.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The cell above prints the validation-accuracy change from stage 1 to the best fine-tuned epoch, and the training curves show it as the step after the dashed line. A small or zero change means the frozen ImageNet features were already sufficient; a gain means adaptation to MRI texture helped.

### 7.6 Model comparison

In [ ]:
# Compare every trained model (headline columns use the clean/leak-free test set)
comparison = compare_models(RESULTS)
display(comparison)
comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(comparison))
err = np.array([comparison.accuracy - comparison.acc_ci_low, comparison.acc_ci_high - comparison.accuracy])
axes[0].bar(x - 0.2, comparison.accuracy, 0.4, yerr=err, capsize=4, label="accuracy (95% CI)")
axes[0].bar(x + 0.2, comparison.f1, 0.4, label="macro-F1")
axes[0].axhline(baseline["leak_free"]["accuracy"], color="grey", ls="--", label="HOG+SVM baseline accuracy")
axes[0].set_xticks(x, comparison.model, rotation=15); axes[0].set_ylim(0.5, 1.02)
axes[0].set_title("Test performance"); axes[0].legend(loc="lower right")
sc = axes[1].scatter(comparison.latency_ms, comparison.f1, s=comparison.params_M * 30 + 40, alpha=0.6)
for _, r in comparison.iterrows():
    axes[1].annotate(r.model, (r.latency_ms, r.f1), textcoords="offset points", xytext=(6, 6))
axes[1].set_xlabel("Latency per image (ms, this machine)"); axes[1].set_ylabel("Macro-F1")
axes[1].set_title("Accuracy vs. speed (bubble size = parameters)")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=130); plt.show()

**How to read it.** The confidence interval on accuracy matters: with a few hundred test images, differences of one or two points between models are within noise, which is why the selection rule below prefers the smaller model when scores are within 0.005 macro-F1 of the best. The dashed line shows what the classical baseline achieves; a deep model is only worth using for the margin above it.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

* **Recall per class (sensitivity)** - of all patients who really have a glioma/meningioma/pituitary tumour, how many does the model catch? A missed tumour delays treatment, so recall on the tumour classes is the clinically most important number.
* **Precision per class** - of the patients labelled with a tumour type, how many really have it? Low precision means unnecessary anxiety, follow-up scans and biopsies, and wasted radiologist time.
* **Macro-F1** - the average of per-class F1 (harmonic mean of precision and recall) with equal weight per class, so a weak small class (no tumour, meningioma) cannot be hidden by a strong large one. It is the main selection metric.
* **Accuracy with a 95% confidence interval** - easy to communicate, but reported with its interval because the test set is small.
* **Confusion matrix** - shows *which* mistakes occur (e.g. meningioma vs. glioma, or tumour vs. no tumour), which matters more clinically than the count of mistakes.
* **Model size and latency** - decide whether the app can run on an ordinary server/CPU and how fast a radiologist gets an answer.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The rule is fixed in advance and implemented in `select_best_model`: **highest macro-F1 on the clean test set; among models within 0.005 of that score, the one with the fewest parameters** (cheaper to serve, and the difference is inside the noise). The cell below applies it to this run, writes `models/best_model.json` (read by the Streamlit app to pre-select its model) and prints the justification.

In [ ]:
best_name = select_best_model(comparison)
json.dump({"name": best_name}, open(MODELS_DIR / "best_model.json", "w"))
best = comparison.set_index("model").loc[best_name]
top = comparison.f1.max()
print(f"Selected model: {best_name}")
print(f"  macro-F1 {best.f1:.3f} (best in table: {top:.3f}), accuracy {best.accuracy:.3f} "
      f"[95% CI {best.acc_ci_low:.3f}-{best.acc_ci_high:.3f}], {best.params_M:.1f} M parameters, "
      f"{best.size_MB:.0f} MB file, {best.latency_ms:.0f} ms per image")
close = comparison[(comparison.f1 >= top - 0.005) & (comparison.model != best_name)]
if len(close):
    print("  Statistically indistinguishable from:", ", ".join(close.model), "- chosen for being smaller.")
print(f"  Reference: HOG+SVM baseline accuracy {baseline['leak_free']['accuracy']:.3f}, macro-F1 {baseline['leak_free']['macro_f1']:.3f}")

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

**Occlusion sensitivity** (`src/explain.py`) is used because it works for every model here, needs no access to internals, and is easy to explain: a grey 32x32 patch is slid over the image and the drop in the predicted class's probability is recorded; regions whose removal hurts most are the regions the model relies on (red in the maps). Below, two correctly classified test images per class from the selected model. A good model's evidence should sit on or around the lesion; evidence on the skull edge or the black background would be a warning sign of shortcut learning.

In [ ]:
# Explainability: occlusion maps for the selected model
best_model = keras.models.load_model(str(MODELS_DIR / f"{best_name}.h5"), compile=False)
predict_fn = lambda batch: best_model.predict(batch, verbose=0)
y_true, y_prob = predict_probs(best_model, test_ds)
y_pred = y_prob.argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(20, 5.6))
col = 0
for cls in range(4):
    for i in np.where((y_true == cls) & (y_pred == cls))[0][:2]:
        img = preprocess_pil(Image.open(test_df.path.iloc[i]))
        heat = occlusion_sensitivity(predict_fn, img, cls, patch=32, stride=24)
        axes[0, col].imshow(img); axes[0, col].set_title(f"{DISPLAY_NAMES[CLASS_NAMES[cls]]}\n{y_prob[i, cls]:.0%}", fontsize=9)
        axes[1, col].imshow(overlay_heatmap(img, heat))
        col += 1
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-0.05, 0.5, "input", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
axes[1, 0].text(-0.05, 0.5, "evidence", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
plt.suptitle(f"Occlusion-sensitivity maps - {best_name} (red = region most important for the prediction)")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "occlusion_maps.png", dpi=130); plt.show()

**How to interpret.** Check whether the red regions cover the lesion or at least brain tissue. Occlusion maps are a coarse tool (32-pixel patches, one class at a time) - they show *where* the model looks, not *why*, and a plausible map does not prove clinical validity. If maps often point at borders/background, treat accuracy figures with extra suspicion given the shortcuts found in Section 4.

#### Error analysis

In [ ]:
# Which test images does the selected model get wrong?
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} of {len(y_true)} test images misclassified ({len(wrong) / len(y_true):.1%})")
if len(wrong):
    pairs = pd.Series([f"{CLASS_NAMES[y_true[i]]} -> {CLASS_NAMES[y_pred[i]]}" for i in wrong]).value_counts()
    display(pairs.rename("count").to_frame())
    shown = wrong[:8]
    fig, axes = plt.subplots(1, len(shown), figsize=(2.6 * len(shown), 3.2), squeeze=False)
    for ax, i in zip(axes[0], shown):
        ax.imshow(Image.open(test_df.path.iloc[i]).convert("RGB")); ax.axis("off")
        ax.set_title(f"true: {CLASS_NAMES[y_true[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})", fontsize=8)
    plt.tight_layout(); plt.show()
    print("Mean confidence when wrong:", round(float(y_prob[wrong].max(1).mean()), 3),
          "| when right:", round(float(y_prob[y_pred == y_true].max(1).mean()), 3))

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.

Deep models are saved in Keras **`.h5`** format (as the brief requires) by `train_model`; the selected model's name is in `models/best_model.json`. The cell below lists the artefacts and, on Colab, bundles them for download.

In [ ]:
# Save the File: list artefacts (models were written by run_experiment) and bundle for download
for f in sorted(MODELS_DIR.glob("*")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:7.1f} MB")
shutil.make_archive("trained_models", "zip", MODELS_DIR)
try:
    from google.colab import files          # only exists on Colab
    files.download("trained_models.zip")
except ImportError:
    print("\nSaved bundle:", Path("trained_models.zip").resolve())

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.

In [ ]:
# Load the File and predict unseen data.
# Reload from disk and predict from the raw image FILES with the app's preprocessing (PIL),
# then compare with the training pipeline's predictions (tf.data) for the same images.
reloaded = keras.models.load_model(str(MODELS_DIR / f"{best_name}.h5"), compile=False)
idx = np.random.default_rng(0).choice(len(test_df), 16, replace=False)
batch = np.stack([preprocess_pil(Image.open(test_df.path.iloc[i])) for i in idx])
app_probs = reloaded.predict(batch, verbose=0)
agree = (app_probs.argmax(1) == y_pred[idx]).mean()
print(f"Reloaded model vs. in-memory pipeline: class agreement {agree:.0%}, "
      f"max probability difference {np.abs(app_probs - y_prob[idx]).max():.4f}")
if agree < 1.0:
    print("WARNING: preprocessing in the app differs from training - investigate before deploying.")
for i, p in list(zip(idx, app_probs))[:5]:
    d = describe_prediction(p)
    print(f"  true = {CLASS_NAMES[test_df.label.iloc[i]]:10s} -> predicted {d['label']:10s} ({d['confidence']:.1%})")

The agreement check matters: the Streamlit app preprocesses uploads with PIL while training used `tf.data`; the anti-aliased bilinear resize in both is designed to make them match, and this cell verifies it on real images.

### Deploying
```bash
pip install -r requirements.txt
# copy models/*.h5 and models/best_model.json (and reports/metrics_*.json, optional) into the repo folders
streamlit run app/streamlit_app.py
```
### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

**What the data audit showed** (fixed facts about this dataset): the labels and files are clean, but the official split leaks - 164 validation/test images (97 validation, 67 test), all gliomas, are variants of scans in another split, so 84% of official test gliomas are not genuinely unseen - and the classes differ in orientation, sequence, framing and compression so strongly that seven global brightness/texture numbers classify about 87% of the official test set correctly (versus 33% for guessing) and 100% of the leaked images. Models on this dataset must therefore be judged on a leak-free split, against a baseline, and ideally on external data.

**What this project delivers:** a leak-free, grouped evaluation protocol; a classical baseline; a custom CNN and three transfer-learning models trained with augmentation, class weights and standard callbacks; a metric-based, rule-driven model selection; occlusion-sensitivity explanations; saved `.h5` models; and a Streamlit app. The cell below writes the data-driven part of the conclusion from this run's results.

In [ ]:
# Data-driven conclusion, generated from this run's results
top = comparison.iloc[0]
print(f"Evaluation protocol: {SPLIT_MODE!r} split, test set of {RESULTS[best_name]['leak_free']['n']} images.\n")
print(f"* Classical baseline (HOG + SVM): accuracy {baseline['leak_free']['accuracy']:.1%}, macro-F1 {baseline['leak_free']['macro_f1']:.3f}.")
for _, r in comparison.iterrows():
    print(f"* {r.model:15s} accuracy {r.accuracy:.1%} (95% CI {r.acc_ci_low:.1%}-{r.acc_ci_high:.1%}), macro-F1 {r.f1:.3f}, "
          f"{r.params_M:.1f} M params, {r.latency_ms:.0f} ms/image")
print(f"\nDeployed model: {best_name}. Best macro-F1 in this run: {top.model} ({top.f1:.3f}).")
gain = comparison.f1.max() - baseline["leak_free"]["macro_f1"]
print(f"Gain of the best deep model over the HOG + SVM baseline: {gain:+.3f} macro-F1.")
weakest = pd.DataFrame(RESULTS[best_name]["leak_free"]["per_class"]).T["recall"].astype(float)
print(f"Weakest class for {best_name}: {weakest.idxmin()} (recall {weakest.min():.1%}, n = {int(RESULTS[best_name]['leak_free']['per_class'][weakest.idxmin()]['support'])}).")
print("\nCaveats: slice-level (not patient-level) split; one dataset from mixed sources; class differences in "
      "acquisition mean accuracy here may not carry over to other hospitals. Not a medical device.")

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***